# RQ1: What Key Factors Lead to Late Deliveries, and How Can We Proactively Identify High-Risk Orders to Improve Customer Satisfaction?

## Importing Libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set()

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import TruncatedSVD
from imblearn.over_sampling import SMOTE

from sklearn.metrics import classification_report, roc_auc_score

## Importing Dataset

In [2]:
data = pd.read_csv("../data/cleaned/cleaned_dataset.csv")

data.head()

,order_id,order_item_id,order_status,order_purchase_timestamp,order_approved_at,order_estimated_delivery_date,is_canceled,customer_id,customer_unique_id,customer_zip_code_prefix,...,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_volume,review_id,review_score,delivered_late,delivery_delay
0,e481f51cbdc54678b7cc49136f2d6af7,1.0,delivered,2017-10-02,2017-10-02,2017-10-18,False,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,3149,...,4.0,500.0,19.0,8.0,13.0,1976.0,a54f0611adc9ed256b57ede6b6eb5114,4.0,False,-8.0
1,53cdb2fc8bc7dce0b6741e2150273451,1.0,delivered,2018-07-24,2018-07-26,2018-08-13,False,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,47813,...,1.0,400.0,19.0,13.0,19.0,4693.0,8d5266042046a06655c8db133d120ba5,4.0,False,-6.0
2,47770eb9100c2d0c44946d9cf07ec65d,1.0,delivered,2018-08-08,2018-08-08,2018-09-04,False,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,1.0,420.0,24.0,19.0,21.0,9576.0,e73b67b67587f7644d5bd1a52deb1b01,5.0,False,-18.0
3,949d5b44dbf5de918fe9c16f97b45f8a,1.0,delivered,2017-11-18,2017-11-18,2017-12-15,False,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,59296,...,3.0,450.0,30.0,10.0,20.0,6000.0,359d03e676b3c069f62cadba8dd3f6e8,5.0,False,-13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,1.0,delivered,2018-02-13,2018-02-13,2018-02-26,False,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,4.0,250.0,51.0,15.0,15.0,11475.0,e50934924e227544ba8246aeb3770dd4,5.0,False,-10.0


In [3]:
data.columns

Index(['order_id', 'order_item_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_estimated_delivery_date', 'is_canceled',
       'customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state', 'customer_lat', 'customer_lng',
       'order_delivered_customer_date', 'is_first_time_buyer', 'seller_id',
       'seller_zip_code_prefix', 'seller_city', 'seller_state', 'seller_lat',
       'seller_lng', 'shipping_limit_date', 'order_delivered_carrier_date',
       'handoff_delay', 'distance_km', 'product_id',
       'product_category_name_english', 'price', 'freight_value',
       'product_name_length', 'product_description_length',
       'product_photos_qty', 'product_weight_g', 'product_length_cm',
       'product_height_cm', 'product_width_cm', 'product_volume', 'review_id',
       'review_score', 'delivered_late', 'delivery_delay'],
      dtype='object')

## Pre-Processing

### Removing Unnecessary Columns

In [4]:
data.drop(columns=['order_id', 
                   'order_item_id', 
                   'order_status', 
                   'order_purchase_timestamp', 
                   'order_approved_at',
                   'order_estimated_delivery_date',
                   'is_canceled',
                   'customer_id',
                   'customer_unique_id',
                   'customer_zip_code_prefix',
                   'customer_state',
                   'customer_lat',
                   'customer_lng',
                   'order_delivered_customer_date',
                   'is_first_time_buyer',
                   'seller_zip_code_prefix',
                   'seller_state',
                   'seller_lat',
                   'seller_lng',
                   'shipping_limit_date',
                   'order_delivered_carrier_date',
                   'price',
                   'freight_value',
                   'product_name_length',
                   'product_description_length',
                   'product_photos_qty',
                   'review_id',
                   'review_score',
                   'product_id',
                   'delivery_delay'], inplace=True)

display(data.head())

display(data.shape)

,customer_city,seller_id,seller_city,handoff_delay,distance_km,product_category_name_english,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_volume,delivered_late
0,sao paulo,3504c0cb71d7fa48d967e0e4c94d59d9,maua,-2.0,18.576110,housewares,500.0,19.0,8.0,13.0,1976.0,False
1,barreiras,289cdb325fb7e7f891c38608bf9e0962,belo horizonte,-4.0,851.495069,perfumery,400.0,19.0,13.0,19.0,4693.0,False
2,vianopolis,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,-5.0,514.410666,auto,420.0,24.0,19.0,21.0,9576.0,False
3,sao goncalo do amarante,66922902710d126a0e7d26b0e3805106,belo horizonte,-1.0,1822.226336,pet_shop,450.0,30.0,10.0,20.0,6000.0,False
4,santo andre,2c9e548be18521d1c43cde1c582c6de8,mogi das cruzes,-5.0,29.676625,stationery,250.0,51.0,15.0,15.0,11475.0,False


(111717, 12)

### Removing NA Rows

In [5]:
data = data.dropna()

display(data.shape)

(108983, 12)

### Variable Selection

In [6]:
X = data.drop(columns=['delivered_late'])
y = data['delivered_late']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### Encoding Variables

In [7]:
categorical_features = ['customer_city',
                        'seller_city', 
                        'seller_id',
                        'product_category_name_english']

numerical_features = ['handoff_delay', 'distance_km', 'product_weight_g',
                      'product_length_cm', 'product_height_cm',
                      'product_width_cm', 'product_volume']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_features),
    ('num', StandardScaler(), numerical_features)
])

# Transform
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

### Getting Feature Names

In [8]:
if hasattr(X_train_encoded, "toarray"):
    X_train_encoded = X_train_encoded.toarray()
    X_test_encoded = X_test_encoded.toarray()

# Get column names
cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)
all_feature_names = list(cat_feature_names) + numerical_features

# Convert to DataFrames
X_train_df = pd.DataFrame(X_train_encoded, columns=all_feature_names)
X_test_df = pd.DataFrame(X_test_encoded, columns=all_feature_names)

### Applying SMOTE

In [9]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_df, y_train)

X_train_resampled.shape, y_train_resampled.shape, X_test_df.shape, y_test.shape, all_feature_names

((163140, 7211),
 (163140,),
 (21797, 7211),
 (21797,),
 ['customer_city_abadia dos dourados',
  'customer_city_abadiania',
  'customer_city_abaete',
  'customer_city_abaetetuba',
  'customer_city_abaiara',
  'customer_city_abaira',
  'customer_city_abare',
  'customer_city_abatia',
  'customer_city_abelardo luz',
  'customer_city_abrantes',
  'customer_city_abre campo',
  'customer_city_abreu e lima',
  'customer_city_acaiaca',
  'customer_city_acailandia',
  'customer_city_acajutiba',
  'customer_city_acarau',
  'customer_city_acari',
  'customer_city_acegua',
  'customer_city_acopiara',
  'customer_city_acreuna',
  'customer_city_acu',
  'customer_city_acucena',
  'customer_city_adamantina',
  'customer_city_adhemar de barros',
  'customer_city_adolfo',
  'customer_city_adrianopolis',
  'customer_city_afogados da ingazeira',
  'customer_city_afonso claudio',
  'customer_city_afranio',
  'customer_city_agisse',
  'customer_city_agrolandia',
  'customer_city_agronomica',
  'customer_c